# 07. LightTools Integration

## LightTools 시뮬 데이터의 두 가지 역할

### 1. 학습용: L_I Target (강도 비교)


### 2. 검증용: PINN vs LT 비교
학습 완료 후 PINN PSF vs LightTools PSF 비교 → 정성적 검증

### 왜 위상 없어도 L_I에 쓸 수 있나?


### 이 노트북의 단계
1. 시뮬레이션 계획 (LHS 샘플링)
2. LightTools 모델 설정 확인 (★ Thin Film 코팅 필수)
3. 배치 시뮬 실행 (COM API 또는 수동)
4. 결과 수집 및 검증
5. TMM+ASM과 강도 일관성 확인 (z=40)
6. L_I target 포맷 → PINN 학습 Stage 3

In [ ]:
import sys, json
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print(f'Project: {PROJECT_ROOT.name}')

---
# Step 1: Simulation Plan (LHS Sampling)

v6 예산: 최대 200 runs

LHS(Latin Hypercube Sampling)로 설계 공간을 효율적으로 탐색합니다.

```
설계변수 4D × 각도 5개 = 조합 수
예: 40 designs × 5 angles = 200 runs (예산 한도)
```

In [ ]:
from backend.data.lhs_sampler import generate_lhs_samples, save_simulation_plan

# ── 시뮬레이션 계획 생성 ──
N_DESIGNS = 40   # 설계변수 조합 수
N_ANGLES = 5     # 각도 수 (0, 10, 20, 30, 40도)

plan = generate_lhs_samples(
    n_samples=N_DESIGNS,
    include_angles=True,
    n_angles=N_ANGLES,
    seed=42,
)

print(f'Design combinations: {N_DESIGNS}')
print(f'Angles: {plan["theta_values"]}')
print(f'Total simulations: {plan["n_total"]} (budget: 200)')
print()

# 설계변수 분포 확인
params = plan['design_params']
names = plan['param_names']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Histograms
for i, (ax, name) in enumerate(zip(axes[0], names[:3])):
    ax.hist(params[:, i], bins=15, color='steelblue', edgecolor='white')
    ax.set_title(f'{name} distribution', fontsize=10)
    ax.set_xlabel(f'{name} (um)')
axes[0, 3] if len(names) > 3 else None
ax = axes[0, 2]

# w2 histogram on remaining subplot
axes[1, 0].hist(params[:, 3], bins=15, color='steelblue', edgecolor='white')
axes[1, 0].set_title(f'{names[3]} distribution', fontsize=10)
axes[1, 0].set_xlabel(f'{names[3]} (um)')

# 2D scatter plots
axes[1, 1].scatter(params[:, 0], params[:, 1], s=20, alpha=0.7, c='steelblue')
axes[1, 1].set_xlabel('delta_bm1')
axes[1, 1].set_ylabel('delta_bm2')
axes[1, 1].set_title('LHS coverage (offsets)', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].scatter(params[:, 2], params[:, 3], s=20, alpha=0.7, c='coral')
axes[1, 2].set_xlabel('w1')
axes[1, 2].set_ylabel('w2')
axes[1, 2].set_title('LHS coverage (widths)', fontsize=10)
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle(f'LHS Sampling: {N_DESIGNS} designs in 4D space', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 처음 5개 config 확인
print('First 5 simulation configs:')
for cfg in plan['all_configs'][:5]:
    print(f'  #{cfg["sim_id"]:3d}: d1={cfg["delta_bm1"]:+6.2f} d2={cfg["delta_bm2"]:+6.2f} '
          f'w1={cfg["w1"]:5.1f} w2={cfg["w2"]:5.1f} theta={cfg["theta_deg"]:4.0f}')

In [ ]:
# ── 시뮬 계획을 JSON으로 저장 (LightTools 배치 실행용) ──
plan_path = PROJECT_ROOT / 'data' / 'lt_checkpoint' / 'simulation_plan.json'
save_simulation_plan(plan, str(plan_path))
print(f'Saved: {plan_path}')
print(f'Total runs: {plan["n_total"]}')

---
# Step 2: LightTools Model Setup Checklist

**LightTools를 열어서 아래 사항을 확인하세요:**

### 필수 확인 항목 (TMM+ASM 일관성)

| 항목 | 필요값 | LT 모델에서 확인 |
|------|--------|------------------|
| Wavelength | 520 nm | Source 설정 |
| AR Layer 1 (SiO2) | 34.6 nm, n=1.46 | Thin film 설정 |
| AR Layer 2 (TiO2) | 25.9 nm, n=2.35 | Thin film 설정 |
| AR Layer 3 (SiO2) | 20.7 nm, n=1.46 | Thin film 설정 |
| AR Layer 4 (TiO2) | 169.5 nm, n=2.35 | Thin film 설정 |
| Cover Glass | 550 um, n=1.52 | Glass 객체 |
| BM1 z-position | 20 um from OPD | BM1 위치 |
| BM2 z-position | 40 um from OPD | BM2 위치 |
| OPD pitch | 72 um | Receiver 설정 |
| OPD pixel width | 10 um | Receiver 설정 |
| Number of pitches | 7 | Receiver 범위 |

### LightTools 객체 이름 확인

아래 코드에서 사용하는 객체 이름을 **실제 LT 모델**에 맞게 수정해야 합니다:

In [ ]:
# ══════════════════════════════════════════════════════════════
# ★ 아래 이름을 실제 LightTools 모델 객체 이름으로 수정하세요 ★
# ══════════════════════════════════════════════════════════════

LT_CONFIG = {
    # LightTools 모델 파일 경로
    'model_path': r'C:\LightTools\Models\UDFPS_Phase1.lts',  # ← 실제 경로로 변경
    
    # BM1 슬릿 객체 이름 (LT 모델에서 확인)
    'bm1_object': 'BM1_Slit',          # ← 실제 이름으로 변경
    'bm1_width_prop': 'ApertureWidth',  # ← 실제 속성명으로 변경
    'bm1_offset_prop': 'XDecenter',     # ← 실제 속성명으로 변경
    
    # BM2 슬릿 객체 이름
    'bm2_object': 'BM2_Slit',          # ← 실제 이름으로 변경
    'bm2_width_prop': 'ApertureWidth',
    'bm2_offset_prop': 'XDecenter',
    
    # 광원 객체 이름
    'source_object': 'Light_Source',    # ← 실제 이름으로 변경
    'source_angle_prop': 'TiltX',       # ← 실제 속성명으로 변경
    
    # OPD 수신기 이름
    'receiver_object': 'OPD_Receiver',  # ← 실제 이름으로 변경
}

print('LightTools Config:')
for k, v in LT_CONFIG.items():
    print(f'  {k}: {v}')
print()
print('⚠ 위 이름들을 실제 LightTools 모델에 맞게 수정하세요!')
print('  LightTools에서: Model > Object List에서 객체 이름 확인')

---
# Step 3: Batch Simulation (COM API)

**실행 전 확인사항:**
1. LightTools가 설치되어 있어야 함 (회사 PC)
2. `pip install pywin32` 설치 필요
3. Step 2의 객체 이름을 실제 모델에 맞게 수정
4. LightTools 모델 파일(.lts) 경로 확인

In [ ]:
# ── Option A: COM API 자동 실행 (회사 GPU PC) ──

RUN_LIGHTTOOLS = False  # ← True로 변경하면 실제 LT 실행

if RUN_LIGHTTOOLS:
    from backend.data.lighttools_runner import LightToolsRunner
    
    runner = LightToolsRunner(
        model_path=LT_CONFIG['model_path'],
        max_retries=3,
        timeout_sec=300,
    )
    
    try:
        # LightTools 연결
        runner.connect()
        print('LightTools connected!')
        
        # 배치 실행 (중간 체크포인트 자동 저장)
        results = runner.run_batch(
            configs=plan['all_configs'],
            output_dir=str(PROJECT_ROOT / 'data' / 'lt_results'),
            checkpoint_every=10,
        )
        
        n_ok = sum(1 for r in results if r.success)
        print(f'\nComplete: {n_ok}/{len(results)} successful')
        
    finally:
        runner.disconnect()
else:
    print('LightTools 실행 비활성화 (RUN_LIGHTTOOLS = False)')
    print('실제 실행하려면 위에서 RUN_LIGHTTOOLS = True로 변경')
    print()
    print('또는 CLI로 실행:')
    print('  python scripts/run_lighttools_batch.py --plan data/lt_checkpoint/simulation_plan.json')

In [ ]:
# ── Option B: 수동 실행 가이드 ──
# LightTools GUI에서 직접 실행할 경우

print('=== 수동 실행 가이드 ===')
print()
print('LightTools GUI에서 수동으로 실행할 경우:')
print()

# 처음 10개만 표시
for cfg in plan['all_configs'][:10]:
    print(f'  Sim #{cfg["sim_id"]:3d}:')
    print(f'    BM1: width={cfg["w1"]:.1f}um, offset={cfg["delta_bm1"]:+.1f}um')
    print(f'    BM2: width={cfg["w2"]:.1f}um, offset={cfg["delta_bm2"]:+.1f}um')
    print(f'    Angle: {cfg["theta_deg"]:.0f} deg')
    print(f'    → Save result as: data/lt_results/sim_{cfg["sim_id"]:04d}.npz')
    print()

print(f'  ... (총 {plan["n_total"]}개)')
print()
print('결과 저장 포맷 (numpy):')
print('  np.savez("sim_0001.npz",')
print('    intensity=I_array,    # 1D intensity at z=0')
print('    x_coords=x_array,     # x coordinates (um)')
print('    psf_7=psf_array,      # 7 OPD pixel values')
print('    delta_bm1=0.0, delta_bm2=0.0, w1=10.0, w2=10.0,')
print('    theta_deg=0.0')
print('  )')

---
# Step 4: Result Validation

시뮬 결과가 올바른지 확인합니다.

In [ ]:
# ── 수집된 결과 확인 ──
lt_dir = PROJECT_ROOT / 'data' / 'lt_results'
result_files = sorted(lt_dir.glob('sim_*.npz'))

if not result_files:
    print(f'No results yet in {lt_dir}')
    print('Run LightTools batch first (Step 3)')
else:
    print(f'Found {len(result_files)} result files')
    
    # Load and visualize first few
    n_show = min(4, len(result_files))
    fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 4))
    if n_show == 1: axes = [axes]
    
    for ax, f in zip(axes, result_files[:n_show]):
        data = np.load(f)
        ax.plot(data['x_coords'], data['intensity'], 'b-', lw=0.8)
        ax.set_title(f'{f.stem}\nd1={float(data["delta_bm1"]):+.1f} '
                     f'w1={float(data["w1"]):.0f} '
                     f'th={float(data["theta_deg"]):.0f}', fontsize=8)
        ax.set_xlabel('x (um)')
        ax.set_ylabel('Intensity')
        ax.grid(True, alpha=0.2)
        
        # Show PSF 7
        if 'psf_7' in data:
            psf = data['psf_7']
            for i in range(7):
                cx = i * 72 + 36
                ax.axvline(x=cx, color='gray', ls=':', alpha=0.3)
    
    plt.suptitle('LightTools Simulation Results', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Basic statistics
    print(f'\nResult statistics:')
    for f in result_files[:5]:
        data = np.load(f)
        I = data['intensity']
        print(f'  {f.stem}: I range=[{I.min():.4f}, {I.max():.4f}], '
              f'mean={I.mean():.4f}, PSF[3]={data["psf_7"][3]:.4f}')

---
# Step 5: Intensity Consistency Check (TMM+ASM vs LT)

**가장 중요한 검증**: LT와 TMM+ASM의 z=40 **강도**가 일치하는지 확인



검증: BM 없는 모델에서 z=40 강도 비교 → 차이 < 5%이면 OK

In [ ]:
# ── TMM+ASM 계산 결과 (우리 파이프라인) ──
from backend.training.loss_functions import ASMIncidentLUT
import torch

asm_lut = ASMIncidentLUT(str(PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'))

# theta=0 에서 |U|^2 (강도)
x_check = torch.linspace(0, 504, 1000)
U_re, U_im = asm_lut.lookup(x_check, torch.zeros(1000))
I_asm = (U_re**2 + U_im**2).numpy()

print('TMM+ASM at z=40, theta=0:')
print(f'  |U|^2 mean = {I_asm.mean():.6f}')
print(f'  |U|^2 std  = {I_asm.std():.6f}  (should be ~0 for plane wave)')
print()
print('이 값을 LightTools z=40 가상 수신기 결과와 비교하세요.')
print('차이가 < 5%이면 일관성 OK.')
print()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x_check.numpy(), I_asm, 'b-', lw=1.5, label='TMM+ASM (our pipeline)')

# If LT z=40 data exists, overlay it
lt_z40_path = lt_dir / 'z40_verification.npz'
if lt_z40_path.exists():
    lt_z40 = np.load(lt_z40_path)
    ax.plot(lt_z40['x'], lt_z40['intensity'], 'r--', lw=1.5, label='LightTools z=40')
    rel_err = np.abs(I_asm - np.interp(x_check.numpy(), lt_z40['x'], lt_z40['intensity'])) / (I_asm + 1e-8)
    ax.set_title(f'Consistency Check: mean rel error = {rel_err.mean()*100:.2f}%')
else:
    ax.set_title('TMM+ASM Intensity at z=40 (overlay LT result for comparison)')

ax.set_xlabel('x (um)')
ax.set_ylabel('|U|^2 (intensity)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n검증 방법:')
print('  1. LightTools에서 BM을 제거하고 z=40에 가상 수신기 배치')
print('  2. theta=0 시뮬 실행 → z=40 강도 프로파일 추출')
print('  3. 결과를 data/lt_results/z40_verification.npz로 저장:')
print('     np.savez("z40_verification.npz", x=x_array, intensity=I_array)')
print('  4. 이 셀을 다시 실행 → 빨간 점선과 파란 실선 비교')

---
# Step 6: L_I Target Dataset Construction

수집된 LT 결과를 PINN 학습용 L_I target으로 변환합니다.

In [ ]:
# ── L_I dataset 생성 ──
if result_files:
    from backend.data.lighttools_runner import LTResultDataset
    
    try:
        dataset = LTResultDataset(str(lt_dir))
        print(f'L_I Dataset loaded: {dataset.n_samples} samples')
        print(f'x range: [{dataset.x_coords.min():.1f}, {dataset.x_coords.max():.1f}] um')
        print(f'x points: {len(dataset.x_coords)}')
        
        # Show random sample
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        indices = dataset.sample_random(6)
        
        for ax, idx in zip(axes.flat, indices):
            cfg, x, I = dataset.get_target(idx)
            ax.plot(x, I, 'b-', lw=0.8)
            ax.set_title(f'd1={cfg["delta_bm1"]:+.1f} d2={cfg["delta_bm2"]:+.1f}\n'
                         f'w1={cfg["w1"]:.0f} w2={cfg["w2"]:.0f} th={cfg["theta_deg"]:.0f}',
                         fontsize=8)
            ax.set_xlabel('x (um)')
            ax.set_ylabel('I(x)')
            ax.grid(True, alpha=0.2)
        
        plt.suptitle('L_I Training Targets (LightTools)', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print('\n사용법 (PINN 학습 시):')
        print('  from backend.data.lighttools_runner import LTResultDataset')
        print('  dataset = LTResultDataset("data/lt_results")')
        print('  cfg, x, I_target = dataset.get_target(idx)')
        print('  # PINN |U(x, z=0)|^2 와 I_target 비교 → L_I loss')
    except FileNotFoundError:
        print('No valid result files found.')
else:
    print('No LT results. Complete Step 3 first.')
    print()
    print('Note: L_I는 선택사항 (v6 Section 6.5)')
    print('L_I 없이도 L_H + L_phase + L_BC로 학습 가능')
    print('L_I는 Phase C-2 (LT 데이터 확보 후) 추가 예정')

---
## Summary

```
전체 워크플로우:

  [Step 1] LHS 샘플링 → simulation_plan.json
      ↓
  [Step 2] LightTools 모델 설정 확인 (AR+CG 일치)
      ↓
  [Step 3] 배치 실행 → data/lt_results/sim_*.npz
      ↓
  [Step 4] 결과 검증 (비정상 제거)
      ↓
  [Step 5] TMM+ASM 일관성 확인 (z=40 비교)
      ↓
  [Step 6] L_I dataset → PINN Stage 3 학습
```

### L_I 학습 활성화
```python
# configs/phase_c_full_gpu.yaml 에서:
curriculum:
  lambda_I: 0.3  # 0.0 → 0.3으로 변경
```